In [1]:
import pandas as pd
import pypsa

# Part A: Analysis of the loaded network

## Import network

In [2]:
# Base case
n0 = pypsa.Network("results/tyndp/NT/networks/pre-1107/solved_2050.nc")

# +100 Mt sequestration limit
# n0 = pypsa.Network("results/tyndp/NT/networks/solved_2050.nc")

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'PyPSA-Eur (tyndp)' has buses, carriers, generators, global_constraints, links, loads, shapes, storage_units, stores, sub_networks


## Buses

In [3]:
n0.buses.carrier.unique()

array(['AC', 'lignite', 'coal', 'oil', 'oil primary', 'uranium', 'none',
       'co2', 'co2 stored', 'co2 sequestered', 'gas', 'hydro-phs',
       'hydro-phs-pure', 'solid biomass', 'H2', 'AC_DRES', 'AC_SRES',
       'H2 tank-storage', 'H2 cavern-storage', 'sng', 'e-liquids',
       'battery', 'home battery', 'exogenous gas demand', 'biogas',
       'shipping oil', 'kerosene for aviation', 'low voltage',
       'EV Street Fixed', 'EV Prosumer Fixed', 'EV Street Commuter',
       'EV Prosumer Commuter', 'EV Street Non-Commuter',
       'EV Prosumer Non-Commuter', 'land transport oil',
       'e-liquids import', 'sng import'], dtype=object)

In [4]:
n0.buses[n0.buses.carrier == "co2"]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,category,substation_lv,substation_off
name,,,,,,,,,,,,,,,,,
co2 atmosphere,1.0,,-5.5,46.0,co2,t_co2,EU,1.0,0.0,inf,Slack,,115,,,NaN,NaN


In [5]:
n0.buses[n0.buses.carrier == "co2 stored"]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,category,substation_lv,substation_off
name,,,,,,,,,,,,,,,,,
co2 stored,1.0,,-5.5,46.0,co2 stored,t_co2,EU,1.0,0.0,inf,Slack,,116,,,NaN,NaN


In [6]:
n0.buses[n0.buses.carrier == "co2 sequestered"]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,category,substation_lv,substation_off
name,,,,,,,,,,,,,,,,,
co2 sequestered,1.0,,-5.5,46.0,co2 sequestered,t_co2,EU,1.0,0.0,inf,Slack,,117,,,NaN,NaN


## Statistics

### Biomass must run condition & balance

In [7]:
n0.generators[n0.generators.carrier == "solid biomass"].e_sum_min.div(1e6).round(1)  # in TWh

name
EU solid biomass    1701.8
Name: e_sum_min, dtype: float64

In [8]:
n0.generators[n0.generators.carrier == "solid biomass"].e_sum_max.div(1e6).round(1)  # in TWh

name
EU solid biomass    1701.8
Name: e_sum_max, dtype: float64

In [9]:
n0.statistics.energy_balance(bus_carrier="solid biomass").div(1e6).round(1).sort_values(ascending=False)

component  carrier                      bus_carrier  
Generator  Solid Biomass                Solid Biomass    1701.8
Link       Other Renewables             Solid Biomass      -0.1
           Biomass to Liquid            Solid Biomass     -41.2
           Biomass to Liquid (CCS)      Solid Biomass    -662.5
Load       Biomass Final Energy Demand  Solid Biomass    -998.0
dtype: float64

### CO2 balance

In [10]:
n0.statistics.energy_balance(bus_carrier="co2").div(1e6).round(1).sort_values(ascending=False)

component  carrier                    bus_carrier
Link       Gas Exogenous Demand       co2            341.9
           Kerosene for Aviation      co2            204.8
           Land Transport Oil Demand  co2            138.3
           Lignite                    co2            111.8
           Coal                       co2             78.1
           Shipping Oil Demand        co2             64.0
           Gas CCGT                   co2             46.0
           Other Non-Renewables       co2             26.9
           Gas Conventional           co2              5.8
           Oil Refining               co2              5.6
           Gas OCGT                   co2              1.7
           Oil (Heavy)                co2              0.4
           Gas CCGT (CCS)             co2              0.2
           Oil (Light)                co2              0.0
           SMR (CCS) (Blue Hydrogen)  co2              0.0
           Biomass to Liquid          co2             -4.8
      

In [11]:
n0.stores[n0.stores.carrier == "co2"]

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,capital_cost,overnight_cost,discount_rate,fom_cost,standing_loss,active,build_year,lifetime,e_nom_opt,marginal_cost_original
name,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,inf,0.0,False,0.0,inf,NaN,-1.0,...,0.0,NaN,NaN,0.0,0.0,True,0,inf,inf,-502.74


Note: the "co2" store is the atmosphere, where "co2" is released into

### CO2 store balance

In [12]:
n0.statistics.energy_balance(bus_carrier="co2 stored").div(1e6).round(2).sort_values(ascending=False)

component  carrier                    bus_carrier
Link       Biomass to Liquid (CCS)    co2 stored     149.66
           Biogas to Gas (CC)         co2 stored      33.54
           Gas CCGT (CCS)             co2 stored       1.54
           SMR (CCS) (Blue Hydrogen)  co2 stored       0.01
           Other Non-Renewables       co2 stored       0.00
           H2 to SNG                  co2 stored      -4.44
           H2 to E-Liquids            co2 stored     -80.57
           co2 sequestered            co2 stored     -99.74
dtype: float64

Insight: The sequestration is "filled up" with BtL CC and BtG CC (due to high co2 price earning)

In [13]:
n0.statistics.energy_balance(bus_carrier="co2 sequestered").div(1e6).round(2).sort_values(ascending=False)

component  carrier          bus_carrier    
Link       co2 sequestered  co2 sequestered    99.74
Store      co2 sequestered  co2 sequestered   -99.74
dtype: float64

In [14]:
n0.links[n0.links.carrier == "biomass to liquid"]

,bus0,bus1,bus2,bus3,type,carrier,efficiency,efficiency2,efficiency3,active,...,under_construction,tags,geometry,dc,underwater_fraction,location,bidirectional,marginal_cost_original,capital_cost_original,reversed
name,,,,,,,,,,,,,,,,,,,,,
EU solid biomass biomass to liquid,EU solid biomass,EU oil,co2 atmosphere,,,biomass to liquid,0.7,-0.1157,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.99449,NaN,False


### SMR CC links

In [15]:
n0.links[n0.links.carrier == "SMR CC"].p_nom

name
BEh2 SMR CCS-2050    10289.855072
DEh2 SMR CCS-2050     7689.855072
ITh2 SMR CCS-2050     2333.333333
NLh2 SMR CCS-2050     3478.260870
GBh2 SMR CCS-2050    11671.014493
FIh2 SMR CCS-2050      483.753623
Name: p_nom, dtype: float64

In [16]:
n0.statistics.energy_balance(bus_carrier="H2", components="Link").div(1e6).round(2)

carrier                      bus_carrier
H2 CCGT                      Hydrogen         -0.02
H2 Cavern Storage Bicharger  Hydrogen         -0.08
H2 Electrolysis              Hydrogen       1562.37
H2 Fuel Cell                 Hydrogen         -0.00
H2 OCGT                      Hydrogen         -0.00
H2 tank-storage charger      Hydrogen        -76.07
H2 tank-storage discharger   Hydrogen         75.31
H2 to E-Liquids              Hydrogen       -436.42
H2 to SNG                    Hydrogen        -28.55
SMR (CCS) (Blue Hydrogen)    Hydrogen          0.06
dtype: float64

### Hydrogen and gas price

In [17]:
n0.statistics.prices(bus_carrier="H2", weighting="load")
# n0.statistics.prices(bus_carrier="H2", groupby="bus_carrier")
# n0.statistics.prices(bus_carrier="H2", groupby_time=False)

name
ALh2             NaN
ATh2        21.32915
BAh2        21.38711
BEh2        21.30672
BGh2        21.37406
              ...   
IB_SKh2E    21.33560
IB_SKh2W    21.41524
PLh2nbc     21.31835
GBh2/INT    21.28559
IB_GRh2P    21.33513
Name: objective, Length: 99, dtype: float64

In [18]:
n0.statistics.energy_balance(bus_carrier="H2").div(1e6).round(2).sort_values(ascending=False)

component  carrier                      bus_carrier
Link       H2 Electrolysis              Hydrogen       1562.37
Generator  H2 Import LH2                Hydrogen        115.25
           H2 Import Pipeline           Hydrogen         87.17
Link       H2 tank-storage discharger   Hydrogen         75.31
           SMR (CCS) (Blue Hydrogen)    Hydrogen          0.06
Generator  Demand Shedding              Hydrogen          0.00
Link       H2 OCGT                      Hydrogen         -0.00
           H2 Fuel Cell                 Hydrogen         -0.00
           H2 CCGT                      Hydrogen         -0.02
           H2 Cavern Storage Bicharger  Hydrogen         -0.08
           H2 to SNG                    Hydrogen        -28.55
           H2 tank-storage charger      Hydrogen        -76.07
           H2 to E-Liquids              Hydrogen       -436.42
Load       H2 Exogenous Demand          Hydrogen      -1299.03
dtype: float64

Gas price insights

In [19]:
n0.statistics.prices(bus_carrier="gas", weighting="load")

name
EU gas    51.11044
Name: objective, dtype: float64

In [20]:
n0.statistics.energy_balance(bus_carrier="gas").div(1e6).round(2).sort_values(ascending=False)

component  carrier                    bus_carrier
Generator  Gas Primary                Gas Primary    1226.91
Link       Biogas to Gas              Gas Primary     630.20
           Biogas to Gas (CC)         Gas Primary     257.51
           SNG to Gas                 Gas Primary      66.05
           SMR (CCS) (Blue Hydrogen)  Gas Primary      -0.08
           Gas OCGT                   Gas Primary      -8.21
           Gas Conventional           Gas Primary     -28.82
           Gas CCGT (CCS)             Gas Primary     -31.16
           Other Non-Renewables       Gas Primary    -192.12
           Gas CCGT                   Gas Primary    -227.70
           Gas Exogenous Demand       Gas Primary   -1692.57
dtype: float64

In [21]:
n0.generators[n0.generators.carrier == "gas"].marginal_cost

name
EU gas    51.110439
Name: marginal_cost, dtype: float64

### CO2 and CO2 store price

In [22]:
n0.statistics.prices(bus_carrier="co2", weighting="load")

name
co2 atmosphere   -502.72989
Name: objective, dtype: float64

In [23]:
n0.statistics.prices(bus_carrier="co2 stored", weighting="load")

name
co2 stored   -465.7894
Name: objective, dtype: float64

### Synfuels balance

In [24]:
n0.statistics.energy_balance(bus_carrier="oil").div(1e6).round(2).sort_values(ascending=False)

component  carrier                    bus_carrier
Link       E-Liquids to Oil           Oil            643.31
           Biomass to Liquid (CCS)    Oil            463.74
           Oil Refining               Oil            407.53
           Biomass to Liquid          Oil             28.87
           Oil (Light)                Oil             -0.00
           Oil (Heavy)                Oil             -1.63
           Shipping Oil Demand        Oil           -242.24
           Land Transport Oil Demand  Oil           -524.00
           Kerosene for Aviation      Oil           -775.59
dtype: float64

### Total system cost

In [25]:
n0.objective / 1e9

np.float64(761.6378955294795)

# Part B: Custom constraints, adjustments and reoptimisation

Work on a copy so `n0` stays untouched for the comparison in Part C.

In [26]:
n1 = n0.copy()

### Adjust SMR marginal costs

In [27]:
# n1.links.loc[n1.links.carrier == "SMR CC", "marginal_cost"] = -60

In [28]:
# n1.links.loc[n1.links.carrier == "SMR CC", "marginal_cost"]

### Adjust Electrolyzer marginal costs

In [29]:
n1.links.loc[n1.links.carrier == "H2 Electrolysis", "marginal_cost"] = 100 # compansate low H2 cost on 168H runs

### Adjust co2 storage quantity

In [30]:
# Adjust the co2 sequestration limit
n1.global_constraints.loc["co2_sequestration_limit", "constant"] = -199e6

In [31]:
# Adjust the co2 sequestration limit
n1.global_constraints #.loc["co2_sequestration_limit", "constant"] 

,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,
co2_synfuels_limit,co2_synfuels,NaN,,co2_emissions,<=,8.578427e+07,0.000000
lv_limit,transmission_volume_expansion_limit,NaN,,"AC, DC",<=,2.564994e+08,0.000000
co2_sequestration_limit,operational_limit,NaN,,co2 sequestered,>=,-1.990000e+08,435.708915


## Add constraint on BECCS and reoptimise (synfuel + BECCS constraint)

In [32]:
def extra(n: pypsa.Network, sns: pd.Index) -> None:
    w = n.snapshot_weightings.generators.loc[sns]
    p = n.model["Link-p"].loc[sns]
    links = n.links
    bio = [("biogas to gas CC", "efficiency2"), ("biomass to liquid CC", "efficiency3")]
    beccs = [(p.loc[:, i] * links.loc[i, eff] * w).sum() for c, eff in bio for i in [links.index[links.carrier == c]]]
    n.model.add_constraints(beccs[0] + beccs[1] <= 85.78e6, name="beccs_limit")
    syn = links.index[links.carrier.isin(["H2 to e-liquids", "H2 to sng"])]
    ccu = (p.loc[:, syn] * -links.loc[syn, "efficiency2"] * w).sum()
    n.model.add_constraints(ccu <= n.global_constraints.at["co2_synfuels_limit", "constant"], name="co2_synfuels")


n1.optimize(solver_name="highs", extra_functionality=extra)
n1.model.constraints["beccs_limit"].dual

/tmp/ipykernel_35487/1397294515.py:13: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n1.optimize(solver_name="highs", extra_functionality=extra)
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 11/11 [00:00<00:00, 519.60it/s]
INFO:linopy.io: Writing time: 0.21s


Running HiGHS 1.15.1 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-pxuqbulu has 361170 rows; 176087 cols; 631756 nonzeros
Coefficient ranges:
  Matrix  [5e-03, 2e+02]
  Cost    [1e-01, 5e+05]
  Bound   [3e-01, 1e+04]
  RHS     [3e-03, 2e+09]
Presolving model
42942 rows, 137683 cols, 256246 nonzeros 0s
27642 rows, 122278 cols, 215514 nonzeros 0s
Dependent equations search running on 25997 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
26537 rows, 120047 cols, 210563 nonzeros 0s
Presolve reductions: rows 26537(-334633); columns 120047(-56040); nonzeros 210563(-421193) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -4.1303842024e+06 Ph1: 473(5746.77); Du: 212(4.13038e+06) 0.5s
      63036     9.6261334393e+11 Pr: 0(0); Du

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 176099 primals, 607738 duals
Objective: 9.63e+11
Solver: highs
Runtime: 1.96s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



  Iteration        Objective     Infeasibilities num(sum)
      63036     9.6261334393e+11 Pr: 1(8.34465e-07); Du: 0(9.42611e-10) 1.9s
      63036     9.6261334393e+11 1.9s

Model name          : linopy-problem-pxuqbulu
Model status        : Optimal
Simplex   iterations: 63036
Objective value     :  9.6261334393e+11
P-D objective error :  1.1476428569e-14
HiGHS run time      :          1.96


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Generator-p_set, Link-fix-p-lower, Link-fix-p-upper, Link-ext-p-lower, Link-ext-p-upper, Link-p-ramp_limit_up, Link-p-ramp_limit_down, Store-fix-e-lower, Store-fix-e-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance, beccs_limit, co2_synfuels were not assigned to the network.


<xarray.DataArray 'dual' ()> Size: 8B
array(-435.70094372)

# Part C: Comparison of the initial and the reoptimised network

Each table shows `initial` (`n0`), `reoptimised` (`n1`) and their difference.

In [33]:
def compare(f) -> pd.DataFrame:
    df = pd.concat({"initial": pd.Series(f(n0)), "reoptimised": pd.Series(f(n1))}, axis=1).fillna(0)
    return df.assign(diff=df.reoptimised - df.initial)

## Biomass balance

In [34]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="solid biomass").div(1e6).round(1).sort_values(ascending=False))

initial  reoptimised  \
component carrier                     bus_carrier                           
Generator Solid Biomass               Solid Biomass   1701.8       1701.8   
Link      Other Renewables            Solid Biomass     -0.1          0.0   
          Biomass to Liquid           Solid Biomass    -41.2       -472.6   
          Biomass to Liquid (CCS)     Solid Biomass   -662.5       -231.2   
Load      Biomass Final Energy Demand Solid Biomass   -998.0       -998.0   

                                                      diff  
component carrier                     bus_carrier           
Generator Solid Biomass               Solid Biomass    0.0  
Link      Other Renewables            Solid Biomass    0.1  
          Biomass to Liquid           Solid Biomass -431.4  
          Biomass to Liquid (CCS)     Solid Biomass  431.3  
Load      Biomass Final Energy Demand Solid Biomass    0.0

## CO2 balance

In [35]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="co2").div(1e6).round(1).sort_values(ascending=False))

initial  reoptimised   diff
component carrier                   bus_carrier                             
Link      Gas Exogenous Demand      co2            341.9        341.9    0.0
          Kerosene for Aviation     co2            204.8        204.8    0.0
          Land Transport Oil Demand co2            138.3        138.3    0.0
          Lignite                   co2            111.8        111.8    0.0
          Coal                      co2             78.1         78.1    0.0
          Shipping Oil Demand       co2             64.0         64.0    0.0
          Gas CCGT                  co2             46.0         46.0    0.0
          Other Non-Renewables      co2             26.9         19.8   -7.1
          Gas Conventional          co2              5.8          5.8    0.0
          Oil Refining              co2              5.6          5.6    0.0
          Gas OCGT                  co2              1.7          1.7    0.0
          Oil (Heavy)               co2              0.4          0.4    0.0
          Gas CCGT (CCS)            co2              0.2          0.2    0.0
          Oil (Light)               co2              0.0          0.0    0.0
          SMR (CCS) (Blue Hydrogen) co2              0.0          6.4    6.4
          Biomass to Liquid         co2             -4.8        -54.7  -49.9
          SNG Import                co2             -8.8        -13.3   -4.5
          Biogas to Gas (CC)        co2            -85.6        -85.6    0.0
          E-Liquids Import          co2            -86.8       -169.8  -83.0
          Biogas to Gas             co2           -127.3       -127.3    0.0
          Biomass to Liquid (CCS)   co2           -226.3        -79.0  147.3
Store     co2                       co2           -485.8       -494.9   -9.1

## CO2 store balance

In [36]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="co2 stored").div(1e6).round(2).sort_values(ascending=False))

initial  reoptimised   diff
component carrier                   bus_carrier                             
Link      Biomass to Liquid (CCS)   co2 stored    149.66        52.23 -97.43
          Biogas to Gas (CC)        co2 stored     33.54        33.55   0.01
          Gas CCGT (CCS)            co2 stored      1.54         1.54   0.00
          SMR (CCS) (Blue Hydrogen) co2 stored      0.01        57.25  57.24
          Other Non-Renewables      co2 stored      0.00         0.04   0.04
          H2 to SNG                 co2 stored     -4.44         0.00   4.44
          H2 to E-Liquids           co2 stored    -80.57         0.00  80.57
          co2 sequestered           co2 stored    -99.74      -144.61 -44.87

In [37]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="co2 sequestered").div(1e6).round(2).sort_values(ascending=False))

,,,initial,reoptimised,diff
component,carrier,bus_carrier,,,
Link,co2 sequestered,co2 sequestered,99.74,144.61,44.87
Store,co2 sequestered,co2 sequestered,-99.74,-144.61,-44.87


In [38]:
compare(lambda n: n.links.loc[n.links.carrier == "biomass to liquid", "p_nom_opt"])

,initial,reoptimised,diff
name,,,
EU solid biomass biomass to liquid,4708.874671,53947.280804,49238.406133


In [39]:
compare(lambda n: n.buses_t.marginal_price.loc[:, "co2 stored"].mean())

,initial,reoptimised,diff
0,-465.789404,-30.088468,435.700935


## SMR CC links

In [40]:
compare(lambda n: n.links.loc[n.links.carrier == "SMR CC", "p_nom_opt"])

,initial,reoptimised,diff
name,,,
BEh2 SMR CCS-2050,10289.855072,10289.855072,0.0
DEh2 SMR CCS-2050,7689.855072,7689.855072,0.0
ITh2 SMR CCS-2050,2333.333333,2333.333333,0.0
NLh2 SMR CCS-2050,3478.260870,3478.260870,0.0
GBh2 SMR CCS-2050,11671.014493,11671.014493,0.0
FIh2 SMR CCS-2050,483.753623,483.753623,0.0


In [41]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="H2", components="Link").div(1e6).round(2))

,,initial,reoptimised,diff
carrier,bus_carrier,,,
H2 CCGT,Hydrogen,-0.02,0.00,0.02
H2 Cavern Storage Bicharger,Hydrogen,-0.08,0.00,0.08
H2 Electrolysis,Hydrogen,1562.37,808.70,-753.67
H2 Fuel Cell,Hydrogen,-0.00,0.00,0.00
H2 OCGT,Hydrogen,-0.00,0.00,0.00
H2 tank-storage charger,Hydrogen,-76.07,-27.67,48.40
H2 tank-storage discharger,Hydrogen,75.31,27.39,-47.92
H2 to E-Liquids,Hydrogen,-436.42,0.00,436.42
H2 to SNG,Hydrogen,-28.55,0.00,28.55


## Hydrogen and gas price

In [42]:
compare(lambda n: n.statistics.prices(bus_carrier="H2", weighting="load"))

,initial,reoptimised,diff
name,,,
ALh2,0.00000,0.00000,0.00000
ATh2,21.32915,135.45163,114.12248
BAh2,21.38711,0.00000,-21.38711
BEh2,21.30672,135.36647,114.05975
BGh2,21.37406,137.85869,116.48463
...,...,...,...
IB_SKh2E,21.33560,135.39443,114.05883
IB_SKh2W,21.41524,135.52443,114.10919
PLh2nbc,21.31835,135.60864,114.29029


In [43]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="H2").div(1e6).round(2).sort_values(ascending=False))

initial  reoptimised  \
component carrier                     bus_carrier                         
Link      H2 Electrolysis             Hydrogen     1562.37       808.70   
Generator H2 Import LH2               Hydrogen      115.25       115.25   
          H2 Import Pipeline          Hydrogen       87.17       158.08   
Link      H2 tank-storage discharger  Hydrogen       75.31        27.39   
          SMR (CCS) (Blue Hydrogen)   Hydrogen        0.06       217.27   
Generator Demand Shedding             Hydrogen        0.00         0.00   
Link      H2 OCGT                     Hydrogen       -0.00         0.00   
          H2 Fuel Cell                Hydrogen       -0.00         0.00   
          H2 CCGT                     Hydrogen       -0.02         0.00   
          H2 Cavern Storage Bicharger Hydrogen       -0.08         0.00   
          H2 to SNG                   Hydrogen      -28.55         0.00   
          H2 tank-storage charger     Hydrogen      -76.07       -27.67   
          H2 to E-Liquids             Hydrogen     -436.42         0.00   
Load      H2 Exogenous Demand         Hydrogen    -1299.03     -1299.03   

                                                     diff  
component carrier                     bus_carrier          
Link      H2 Electrolysis             Hydrogen    -753.67  
Generator H2 Import LH2               Hydrogen       0.00  
          H2 Import Pipeline          Hydrogen      70.91  
Link      H2 tank-storage discharger  Hydrogen     -47.92  
          SMR (CCS) (Blue Hydrogen)   Hydrogen     217.21  
Generator Demand Shedding             Hydrogen       0.00  
Link      H2 OCGT                     Hydrogen       0.00  
          H2 Fuel Cell                Hydrogen       0.00  
          H2 CCGT                     Hydrogen       0.02  
          H2 Cavern Storage Bicharger Hydrogen       0.08  
          H2 to SNG                   Hydrogen      28.55  
          H2 tank-storage charger     Hydrogen      48.40  
          H2 to E-Liquids             Hydrogen     436.42  
Load      H2 Exogenous Demand         Hydrogen       0.00

In [44]:
compare(lambda n: n.statistics.prices(bus_carrier="gas", weighting="load"))

,initial,reoptimised,diff
name,,,
EU gas,51.11044,51.11044,0.0


In [45]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="gas").div(1e6).round(2).sort_values(ascending=False))

initial  reoptimised    diff
component carrier                   bus_carrier                              
Generator Gas Primary               Gas Primary  1226.91      1491.70  264.79
Link      Biogas to Gas             Gas Primary   630.20       630.10   -0.10
          Biogas to Gas (CC)        Gas Primary   257.51       257.61    0.10
          SNG to Gas                Gas Primary    66.05        66.05    0.00
          SMR (CCS) (Blue Hydrogen) Gas Primary    -0.08      -314.89 -314.81
          Gas OCGT                  Gas Primary    -8.21        -8.21    0.00
          Gas Conventional          Gas Primary   -28.82       -28.82    0.00
          Gas CCGT (CCS)            Gas Primary   -31.16       -31.16    0.00
          Other Non-Renewables      Gas Primary  -192.12      -142.16   49.96
          Gas CCGT                  Gas Primary  -227.70      -227.64    0.06
          Gas Exogenous Demand      Gas Primary -1692.57     -1692.57    0.00

## CO2 and CO2 store price

In [46]:
compare(lambda n: n.statistics.prices(bus_carrier="co2", weighting="load"))

,initial,reoptimised,diff
name,,,
co2 atmosphere,-502.72989,-502.72989,0.0


In [47]:
compare(lambda n: n.statistics.prices(bus_carrier="co2 stored", weighting="load"))

,initial,reoptimised,diff
name,,,
co2 stored,-465.7894,-30.08847,435.70093


## Synfuels balance

In [48]:
compare(lambda n: n.statistics.energy_balance(bus_carrier="oil").div(1e6).round(2).sort_values(ascending=False))

initial  reoptimised    diff
component carrier                   bus_carrier                              
Link      E-Liquids to Oil          Oil           643.31       643.31    0.00
          Biomass to Liquid (CCS)   Oil           463.74       161.85 -301.89
          Oil Refining              Oil           407.53       407.49   -0.04
          Biomass to Liquid         Oil            28.87       330.80  301.93
          Oil (Light)               Oil            -0.00         0.00    0.00
          Oil (Heavy)               Oil            -1.63        -1.63    0.00
          Shipping Oil Demand       Oil          -242.24      -242.24    0.00
          Land Transport Oil Demand Oil          -524.00      -524.00    0.00
          Kerosene for Aviation     Oil          -775.59      -775.59    0.00

## Total system cost

In [49]:
compare(lambda n: n.objective / 1e9)

,initial,reoptimised,diff
0,761.637896,962.613344,200.975448


# Test area

In [50]:
n1.links[n1.links.carrier == "H2 Electrolysis"]

,bus0,bus1,bus2,bus3,type,carrier,efficiency,efficiency2,efficiency3,active,...,under_construction,tags,geometry,dc,underwater_fraction,location,bidirectional,marginal_cost_original,capital_cost_original,reversed
name,,,,,,,,,,,,,,,,,,,,,
DE00-DEh2 H2 Z2 Electrolysis-2050,DE00,DEh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
AT00-ATh2 H2 Z2 Electrolysis-2050,AT00,ATh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
BE00-BEh2 H2 Z2 Electrolysis-2050,BE00,BEh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
BG00-BGh2 H2 Z2 Electrolysis-2050,BG00,BGh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
CH00-CHh2 H2 Z2 Electrolysis-2050,CH00,CHh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SK00 DRES Electrolysis SKh2W-2050,SK00 DRES,SKh2W,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
DE00 SRES Electrolysis-2050,DE00 SRES,DEh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False
ES00 SRES Electrolysis-2050,ES00 SRES,ESh2,,,,H2 Electrolysis,0.74,1.0,1.0,True,...,NaN,,,NaN,NaN,,NaN,0.0,NaN,False


In [51]:
n1.links[n1.links.carrier == "SMR CC"]

,bus0,bus1,bus2,bus3,type,carrier,efficiency,efficiency2,efficiency3,active,...,under_construction,tags,geometry,dc,underwater_fraction,location,bidirectional,marginal_cost_original,capital_cost_original,reversed
name,,,,,,,,,,,,,,,,,,,,,
BEh2 SMR CCS-2050,EU gas,BEh2,co2 atmosphere,co2 stored,,SMR CC,0.69,0.0202,0.1818,True,...,NaN,,,NaN,NaN,,NaN,3.283293,NaN,False
DEh2 SMR CCS-2050,EU gas,DEh2,co2 atmosphere,co2 stored,,SMR CC,0.69,0.0202,0.1818,True,...,NaN,,,NaN,NaN,,NaN,3.283293,NaN,False
ITh2 SMR CCS-2050,EU gas,ITh2,co2 atmosphere,co2 stored,,SMR CC,0.69,0.0202,0.1818,True,...,NaN,,,NaN,NaN,,NaN,3.283293,NaN,False
NLh2 SMR CCS-2050,EU gas,NLh2,co2 atmosphere,co2 stored,,SMR CC,0.69,0.0202,0.1818,True,...,NaN,,,NaN,NaN,,NaN,3.283293,NaN,False
GBh2 SMR CCS-2050,EU gas,GBh2,co2 atmosphere,co2 stored,,SMR CC,0.69,0.0202,0.1818,True,...,NaN,,,NaN,NaN,,NaN,3.283293,NaN,False
FIh2 SMR CCS-2050,EU gas,FIh2,co2 atmosphere,co2 stored,,SMR CC,0.69,0.0202,0.1818,True,...,NaN,,,NaN,NaN,,NaN,3.283293,NaN,False
